# Premier League Prediction

## Configuration Constants

In [1]:
# Imports used across the notebook for data loading, feature engineering, and modeling.
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Folder that contains the training data files.
DATA_ROOT = Path("../data")

In [68]:
ROLLING_WINDOW = 5

COUNTRY_CODES = {
    "eng",
    "es",
    "de",
    "fr",
    "it",
    "nl",
    "pt",
    "ru",
    "ua",
    "gr",
    "be",
    "at",
    "hr",
    "rs",
    "tr",
    "ch",
    "dk",
    "no",
    "se",
    "pl",
    "cz",
    "sk",
}

TEAM_ALIASES = {
    "man city": "manchester city",
    "man utd": "manchester united",
    "manchester utd": "manchester united",
    "spurs": "tottenham hotspur",
    "tottenham": "tottenham hotspur",
    "wolves": "wolverhampton wanderers",
    "leicester": "leicester city",
    "norwich": "norwich city",
    "west ham": "west ham united",
    "newcastle": "newcastle united",
    "brighton": "brighton and hove albion",
}

COMP_WEIGHTS = {
    "ucl": {"group": 1.0, "knockout": 2.0, "late": 3.0},
    "uel": {"group": 0.8, "knockout": 1.5, "late": 2.0},
    "fa": {"early": 0.5, "late": 1.5},
    "carabao": {"early": 0.3, "late": 1.0},
}

TEAM_BOOST_OVERRIDES = {
    "liverpool": 0.40,
    "manchester city": 0.00,
}

CONTEXT_COLS = {
    "Possession": ["Possession"],
    "Shot_Creating_Actions": ["Shot_Creating_Actions", "SCA"],
    "Successful_Dribbles": ["Successful_Dribbles", "Dribbles"],
    "Final_Third_Entries": ["Final_Third_Entries"],
    "Final_Third_Entries_Allowed": ["Final_Third_Entries_Allowed"],
    "Aerial_Battles_Won_Pct": ["Aerial_Battles_Won%", "Aerial_Battles_Won_Pct"],
    "Save_Pct": ["Save%", "Save_Pct"],
    "PPDA": ["PPDA"],
    "Allowed_PPDA": ["Allowed_PPDA"],
}

## Utility Functions

In [69]:
# Print a simple notebook log message.
def log(msg: str) -> None:
    print(f"[INFO] {msg}")


# Find the first column name that matches one of the candidates.
def find_col(df: pd.DataFrame, *candidates: str) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


# Normalize team names so aliases and country prefixes match.
def normalize_team(name) -> str:
    s = re.sub(r"[^a-z0-9\s]", " ", str(name).strip().lower())
    tokens = [t for t in s.split() if t]
    if tokens and tokens[0] in COUNTRY_CODES:
        tokens = tokens[1:]
    if tokens and tokens[-1] in COUNTRY_CODES:
        tokens = tokens[:-1]
    s = " ".join(tokens)
    return TEAM_ALIASES.get(s, s)


# Return the column if it exists, otherwise fill with NaN values.
def col_or_nan(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return df[col]
    return pd.Series(np.nan, index=df.index)

## Data Loading Functions

In [70]:
# Load the league data and merge the extra copy into it.
def load_premier_league(base: Path, additional: Path) -> pd.DataFrame:
    dfs = [pd.read_csv(base / "premier_league.csv", low_memory=False)]

    extra = additional / "premier_league.csv"
    dfs.append(pd.read_csv(extra, low_memory=False))
    log("Loaded additional_data/premier_league.csv")

    common = list(set.intersection(*[set(d.columns) for d in dfs]))
    merged = (
        pd.concat([d[common] for d in dfs], ignore_index=True).drop_duplicates().copy()
    )
    log(f"Raw rows after concat + dedup: {len(merged):,}")
    return merged


# Convert the venue/home-away field into a simple h/a label.
def parse_home_away(df: pd.DataFrame) -> pd.Series:
    for col in ("h_a", "side"):
        if col in df.columns:
            return df[col].astype(str).str.lower().str.strip()
    if "Venue" in df.columns:
        return (
            df["Venue"]
            .astype(str)
            .str.lower()
            .str.strip()
            .map({"home": "h", "away": "a"})
        )
    return pd.Series(["a"] * len(df), index=df.index)


# Read the position table and turn it into a team-to-rank lookup.
def load_position_map(path: Path) -> dict[str, float]:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    team_col = find_col(df, "team", "teamid", "club")
    pos_col = find_col(df, "position", "pos", "rank")

    if not team_col or not pos_col:
        raise ValueError(f"{path} must have Team and Position columns")

    df["team_norm"] = df[team_col].map(normalize_team)
    df["pos"] = pd.to_numeric(df[pos_col], errors="coerce")
    df = df.dropna(subset=["team_norm", "pos"])
    return dict(zip(df["team_norm"], df["pos"]))


# Infer the competition stage from the row text and date.
def _infer_stage(comp: str, stage_text: str, dt: pd.Timestamp) -> str:
    s = str(stage_text).lower()
    if any(
        k in s for k in ["semi", "final", "quarter", "qf", "sf", "r16", "round of 16"]
    ):
        return (
            "late"
            if any(k in s for k in ["semi", "final", "quarter", "qf", "sf"])
            else "knockout"
        )
    if "group" in s:
        return "group"
    month = dt.month if pd.notna(dt) else 1
    if comp in ("ucl", "uel"):
        return (
            "group"
            if month in [9, 10, 11, 12]
            else ("late" if month == 8 else "knockout")
        )
    if comp in ("fa", "carabao"):
        return "late" if month >= 3 else "early"
    return "early"


# Turn cup fixtures into weighted fatigue events for each team.
def load_competition_matches(path: Path, comp: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)
    df.columns = df.columns.str.strip()

    date_col = find_col(df, "date")
    home_col = find_col(df, "home team", "home_team", "home")
    away_col = find_col(df, "away team", "away_team", "away")
    stage_col = find_col(df, "stage", "round")

    if not all([date_col, home_col, away_col]):
        raise ValueError(f"{path.name} is missing required columns")

    # Support mixed date styles like YYYY-MM-DD, May 30 2015, and April 18-19 2015.
    date_text = df[date_col].astype(str).str.strip()
    date_text = date_text.str.replace(
        r"([A-Za-z]+\s+\d{1,2})-\d{1,2}(\s+\d{4})", r"\1\2", regex=True
    )
    df["_date"] = pd.to_datetime(date_text, errors="coerce")
    df["_stage"] = df[stage_col] if stage_col else ""

    rows = []
    for _, row in df.iterrows():
        if pd.isna(row["_date"]):
            continue
        stage = _infer_stage(comp, row["_stage"], row["_date"])
        weight = COMP_WEIGHTS.get(comp, {}).get(stage, 0.0)
        for col in (home_col, away_col):
            rows.append(
                {
                    "team": normalize_team(row[col]),  # type: ignore
                    "date": row["_date"],
                    "weight": weight,
                }
            )

    return pd.DataFrame(rows, columns=["team", "date", "weight"])

## Basic Feature Engineering

In [71]:
# Attach a fatigue score based on past competition workload.
def attach_fatigue(df: pd.DataFrame, comp_df: pd.DataFrame) -> pd.DataFrame:
    comp_df = (
        comp_df.dropna(subset=["team", "date"]).sort_values(["team", "date"]).copy()
    )
    comp_df["date"] = pd.to_datetime(comp_df["date"], errors="coerce")
    comp_df = comp_df.dropna(subset=["date"])
    comp_df["date"] = comp_df["date"].astype("datetime64[ns]")
    comp_df["weight"] = pd.to_numeric(comp_df["weight"], errors="coerce").fillna(0.0)
    comp_df["cum_weight"] = comp_df.groupby("team")["weight"].cumsum()

    left = (
        df.sort_values(["Team", "Date"])
        .reset_index()
        .rename(columns={"index": "_orig_idx"})
    )
    left["Team"] = left["Team"].astype(str)
    left["Date"] = pd.to_datetime(left["Date"], errors="coerce")
    left["Date"] = left["Date"].astype("datetime64[ns]")

    right = comp_df.rename(columns={"team": "Team"}).sort_values(["Team", "date"])
    right["Team"] = right["Team"].astype(str)

    left["fatigue_score"] = 0.0
    left_valid = left[left["Date"].notna()].sort_values(["Team", "Date"]).copy()
    if not left_valid.empty and not right.empty:
        merged_parts = []
        for team, left_team in left_valid.groupby("Team", sort=False):
            right_team = right[right["Team"].eq(team)][["date", "cum_weight"]]
            if right_team.empty:
                continue
            left_team = left_team.sort_values("Date")
            right_team = right_team.sort_values("date")
            m = pd.merge_asof(
                left_team,
                right_team,
                left_on="Date",
                right_on="date",
                direction="backward",
                allow_exact_matches=False,
            )
            merged_parts.append(m)

        if merged_parts:
            merged = pd.concat(merged_parts, axis=0)
            left.loc[merged.index, "fatigue_score"] = (
                merged["cum_weight"].fillna(0.0).to_numpy()
            )

    df["fatigue_score"] = left.sort_values("_orig_idx")["fatigue_score"].to_numpy()
    log(f"Fatigue scores attached from {len(comp_df):,} competition rows")
    return df


# Build a rolling mean using only earlier matches.
def _lagged_rolling_mean(series: pd.Series, window: int) -> pd.Series:
    return series.shift(1).rolling(window, min_periods=1).mean()


# Create lagged rolling features for the main performance columns.
def build_rolling_features(
    df: pd.DataFrame, window: int = ROLLING_WINDOW
) -> pd.DataFrame:
    df = df.sort_values(["Team", "Date"]).copy()
    grp = df.groupby("Team", group_keys=False)
    roll = lambda col: grp[col].transform(lambda s: _lagged_rolling_mean(s, window))

    df[f"rolling_xG_{window}"] = roll("xG")
    df[f"rolling_xGA_{window}"] = roll("xGA")
    df[f"rolling_scored_{window}"] = roll("_scored")
    df[f"rolling_conceded_{window}"] = roll("_conceded")
    df[f"rolling_win_rate_{window}"] = roll("_win_val")
    df[f"rolling_ppda_{window}"] = roll("_ppda")
    if "_xpts" in df.columns:
        df[f"rolling_xpts_{window}"] = roll("_xpts")

    log(f"Rolling features built (window={window})")
    return df


# Add league position features for the team and opponent.
def attach_position_features(df: pd.DataFrame, pos_map: dict) -> pd.DataFrame:
    df = df.copy()
    df["team_position"] = df["Team"].map(pos_map)
    df["opponent_position"] = df["Opponent"].map(pos_map)
    df["position_gap"] = df["opponent_position"] - df["team_position"]
    return df


# Add a head-to-head win rate for the two teams in a fixture.
def attach_h2h_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("Date").copy()
    df["_h2h_key"] = df[["Team", "Opponent"]].apply(
        lambda r: "|".join(sorted([r["Team"], r["Opponent"]])), axis=1
    )
    df["_h2h_win"] = (df["Result"] == "w").astype(float)
    df["h2h_win_rate"] = (
        df.groupby("_h2h_key")["_h2h_win"]
        .transform(lambda s: s.shift(1).expanding().mean())
        .fillna(0.5)
    )
    df.drop(columns=["_h2h_key", "_h2h_win"], inplace=True)
    log("H2H win rate attached")
    return df


# Copy any matching context columns into the canonical names.
def attach_context_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    added = []
    for canonical, candidates in CONTEXT_COLS.items():
        src = find_col(df, *candidates)
        if src is not None:
            df[canonical] = pd.to_numeric(df[src], errors="coerce")
            added.append(canonical)
    if added:
        log(f"Context features: {added}")
    return df, added

## Style Analysis and Style Calculations

In [72]:
# Classify how a team plays from its recent matches.
def classify_team_style(df: pd.DataFrame, team: str, window: int = 10) -> dict:
    team_df = df[df["Team"].eq(team)].sort_values("Date").tail(window)
    if team_df.empty:
        return {
            "possession_heavy": False,
            "defensive": False,
            "counter_prone": False,
            "possession_pct": 50.0,
            "ppda": 6.0,
            "defensive_actions": 10.0,
            "clean_sheet_rate": 0.3,
        }

    avg_possession = pd.to_numeric(
        col_or_nan(team_df, "Possession"), errors="coerce"
    ).mean()
    avg_ppda = pd.to_numeric(col_or_nan(team_df, "PPDA"), errors="coerce").mean()
    avg_defensive_actions = pd.to_numeric(
        col_or_nan(team_df, "Defensive_Actions"), errors="coerce"
    ).mean()
    clean_sheet_col = col_or_nan(team_df, "Clean_Sheet")
    avg_clean_sheets = (
        (clean_sheet_col == 1).sum() / len(team_df) if len(team_df) > 0 else 0.0
    )

    avg_possession = avg_possession if pd.notna(avg_possession) else 50.0
    avg_ppda = avg_ppda if pd.notna(avg_ppda) else 6.0
    avg_defensive_actions = (
        avg_defensive_actions if pd.notna(avg_defensive_actions) else 10.0
    )

    return {
        "possession_heavy": avg_possession > 52,
        "defensive": avg_defensive_actions > 15,
        "counter_prone": avg_ppda < 5.5,
        "possession_pct": int(avg_possession),
        "ppda": avg_ppda,
        "defensive_actions": avg_defensive_actions,
        "clean_sheet_rate": float(avg_clean_sheets),
    }


# Summarize recent results and scoring form for a team.
def calculate_recent_form(df: pd.DataFrame, team: str, window: int = 5) -> dict:
    team_df = df[df["Team"].eq(team)].sort_values("Date").copy()
    if len(team_df) < 1:
        return {"points_per_game": 0.0, "streak": 0, "recent_xg": 0.0}

    team_df["_win_val"] = team_df["Result"].map({"w": 1.0, "d": 0.5, "l": 0.0})
    recent = team_df.tail(window)

    points = recent["_win_val"].sum()
    points_per_game = points / len(recent) if len(recent) > 0 else 0.0
    recent_xg = pd.to_numeric(col_or_nan(recent, "xG"), errors="coerce").mean()
    recent_goals = pd.to_numeric(col_or_nan(recent, "Scored"), errors="coerce").mean()

    streak = 0
    for result in recent["Result"].iloc[::-1]:
        if result == "w":
            streak += 1
        elif result == "l":
            streak -= 1
        else:
            break

    return {
        "points_per_game": points_per_game,
        "streak": streak,
        "recent_xg": recent_xg if pd.notna(recent_xg) else 0.0,
        "recent_goals": recent_goals if pd.notna(recent_goals) else 0.0,
    }


# Estimate the opponent strength from recent results.
def calculate_opponent_strength(df: pd.DataFrame, opponent: str) -> float:
    opp_df = df[df["Team"].eq(opponent)].sort_values("Date").tail(10)
    if opp_df.empty:
        return 0.5
    opp_df["_win_val"] = opp_df["Result"].map({"w": 1.0, "d": 0.5, "l": 0.0})
    return float(opp_df["_win_val"].mean())


# Turn league positions into a simple fixture importance score.
def calculate_fixture_importance(
    team_pos: float, opponent_pos: float, row: pd.Series
) -> float:
    if pd.isna(team_pos) or pd.isna(opponent_pos):
        return 0.5

    importance = 0.0
    if team_pos <= 2:
        importance += 0.7
    if team_pos >= 16:
        importance += 0.8
    if 3 <= team_pos <= 7:
        importance += 0.5

    pos_gap = abs(team_pos - opponent_pos)
    if opponent_pos <= 6:
        importance += 0.4
    if pos_gap <= 2:
        importance += 0.3

    return min(1.0, importance)


# Estimate squad depth from the spread of recent formations.
def estimate_squad_depth(df: pd.DataFrame, team: str, window: int = 15) -> float:
    team_df = df[df["Team"].eq(team)].sort_values("Date").tail(window)
    if len(team_df) < 3:
        return 0.5
    formations = col_or_nan(team_df, "Formation").dropna().nunique()
    return min(1.0, formations / max(1, len(team_df)))


# Return how the team's style should shape expected attacking output.
def style_xg_expectation(team_style: dict) -> float:
    if team_style.get("counter_prone"):
        return 0.75
    if team_style.get("possession_heavy"):
        return 1.1
    return 1.0


# Return how the team's style should shape expected defensive output.
def style_defense_expectation(team_style: dict) -> float:
    if team_style.get("defensive"):
        return 0.75
    if team_style.get("counter_prone"):
        return 1.05
    return 1.0


# Compare two styles and produce a small interaction adjustment.
def opponent_style_interaction(team_style: dict, opp_style: dict) -> float:
    adjustment = 0.0
    if team_style.get("counter_prone") and opp_style.get("possession_heavy"):
        adjustment += 0.1
    if team_style.get("possession_heavy") and opp_style.get("defensive"):
        adjustment -= 0.08
    if team_style.get("defensive") and opp_style.get("possession_heavy"):
        adjustment += 0.06
    if team_style.get("defensive") and opp_style.get("counter_prone"):
        adjustment -= 0.07
    return adjustment


# Weight context stats according to the team's style.
def calculate_style_weighted_context(row: pd.Series, team_style: dict) -> dict:
    return {
        "Possession": 0.8 if not team_style.get("possession_heavy") else 1.3,
        "Shot_Creating_Actions": 1.0,
        "Successful_Dribbles": 1.2 if team_style.get("counter_prone") else 0.9,
        "Final_Third_Entries": 1.1 if team_style.get("possession_heavy") else 0.8,
        "Final_Third_Entries_Allowed": 1.2 if team_style.get("defensive") else 0.9,
        "Aerial_Battles_Won_Pct": 1.2 if team_style.get("defensive") else 0.9,
        "Save_Pct": 1.3 if team_style.get("defensive") else 0.85,
        "PPDA": 1.1 if team_style.get("defensive") else 0.9,
    }

## Advanced Feature Engineering

In [73]:
DEFAULT_POSSESSION_PCT = 50.0
DEFAULT_PPDA = 6.0


# Read a numeric value from a row with a fallback default.
def _row_numeric(row: pd.Series, key: str, default: float = np.nan) -> float:
    value = pd.to_numeric(pd.Series([row.get(key, default)]), errors="coerce").iloc[0]
    if pd.isna(value):
        return default
    return float(value)


# Scale raw context values using the style weights.
def _style_weighted_context_values(
    row: pd.Series, team_style: dict
) -> dict[str, float]:
    weights = calculate_style_weighted_context(row, team_style)

    possession = _row_numeric(row, "Possession")
    sca = _row_numeric(row, "Shot_Creating_Actions")
    dribbles = _row_numeric(row, "Successful_Dribbles")
    final_third_entries = _row_numeric(row, "Final_Third_Entries")

    return {
        "Possession_Style_Weighted": (
            possession * weights.get("Possession", 1.0)
            if pd.notna(possession)
            else np.nan
        ),
        "SCA_Style_Weighted": (
            sca * weights.get("Shot_Creating_Actions", 1.0) if pd.notna(sca) else np.nan
        ),
        "Dribbles_Style_Weighted": (
            dribbles * weights.get("Successful_Dribbles", 1.0)
            if pd.notna(dribbles)
            else np.nan
        ),
        "Final_Third_Style_Weighted": (
            final_third_entries * weights.get("Final_Third_Entries", 1.0)
            if pd.notna(final_third_entries)
            else np.nan
        ),
        "Possession_Style_Delta": (
            possession - float(team_style.get("possession_pct", DEFAULT_POSSESSION_PCT))
            if pd.notna(possession)
            else np.nan
        ),
    }


# Adjust rolling attack and defense numbers using team style.
def _style_adjusted_rolling_values(
    row: pd.Series, team_style: dict, rolling_window: int
) -> dict[str, float]:
    rolling_xg = _row_numeric(row, f"rolling_xG_{rolling_window}")
    rolling_xga = _row_numeric(row, f"rolling_xGA_{rolling_window}")

    xg_expectation = style_xg_expectation(team_style)
    xga_expectation = style_defense_expectation(team_style)

    xg_adjusted = (
        rolling_xg / max(xg_expectation, 1e-6) if pd.notna(rolling_xg) else np.nan
    )
    xga_adjusted = rolling_xga * xga_expectation if pd.notna(rolling_xga) else np.nan

    return {
        "rolling_xG_style_adj": xg_adjusted,
        "rolling_xGA_style_adj": xga_adjusted,
        "xG_diff": (
            (xg_adjusted - xga_adjusted)
            if pd.notna(xg_adjusted) and pd.notna(xga_adjusted)
            else np.nan
        ),
    }


# Estimate how referee tendencies affect the match.
def _referee_style_impact_for_row(row: pd.Series) -> float:
    is_home = str(row.get("home_advantage", "a")).lower() == "h"
    foul_col = "Home_Fouls" if is_home else "Away_Fouls"

    team_fouls = _row_numeric(row, foul_col, default=0.0)
    ppda_value = _row_numeric(row, "PPDA", default=DEFAULT_PPDA)
    referee_bias = _row_numeric(row, "Referee_Bias_Score", default=0.0)

    aggression = team_fouls + max(0.0, 8.0 - ppda_value)
    return referee_bias * (1.0 + aggression / 10.0)


# Combine form and opponent strength into momentum features.
def _momentum_features_for_row(
    row: pd.Series, rolling_window: int, recent_form: dict, opponent_strength: float
) -> dict[str, float]:
    rolling_points = _row_numeric(row, f"rolling_win_rate_{rolling_window}")
    if pd.notna(rolling_points):
        rolling_points *= 3.0
    else:
        rolling_points = recent_form["points_per_game"] * 3.0

    return {
        "form_vs_strength": recent_form["points_per_game"] * opponent_strength,
        "momentum": rolling_points + recent_form["streak"],
    }


# Add the style-based engineered features to the match data.
def attach_style_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(["Date", "Team", "Opponent"]).copy()
    w = ROLLING_WINDOW
    feature_rows = []

    for _, row in out.iterrows():
        current_date = row["Date"]
        history = out[out["Date"] < current_date]

        team = row["Team"]
        opponent = row["Opponent"]

        team_style = classify_team_style(history, team, window=10)
        opp_style = classify_team_style(history, opponent, window=10)
        recent_form = calculate_recent_form(history, team, window=5)
        opp_strength = calculate_opponent_strength(history, opponent)

        _ = opponent_style_interaction(team_style, opp_style)
        _ = estimate_squad_depth(history, team, window=15)

        row_features = {}
        row_features.update(_style_weighted_context_values(row, team_style))
        row_features.update(_style_adjusted_rolling_values(row, team_style, w))
        row_features.update(
            _momentum_features_for_row(row, w, recent_form, opp_strength)
        )
        row_features["referee_style_impact"] = _referee_style_impact_for_row(row)

        feature_rows.append(row_features)

    engineered = pd.DataFrame(feature_rows, index=out.index)
    out = pd.concat([out, engineered], axis=1)
    log("Style engineered features attached")
    return out


# Build referee-related bias features from card and penalty patterns.
def attach_referee_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(["Date", "Team", "Opponent"]).copy()
    out["_ref"] = (
        out["Referee"].astype(str).str.lower().str.strip().replace("", "unknown")
    )
    home_mask = out["home_advantage"].astype(str).str.lower().eq("h")

    home_y = pd.to_numeric(col_or_nan(out, "Home_Yellow"), errors="coerce")
    home_r = pd.to_numeric(col_or_nan(out, "Home_Red"), errors="coerce")
    away_y = pd.to_numeric(col_or_nan(out, "Away_Yellow"), errors="coerce")
    away_r = pd.to_numeric(col_or_nan(out, "Away_Red"), errors="coerce")

    home_cards = home_y.fillna(0.0) + 2.0 * home_r.fillna(0.0)
    away_cards = away_y.fillna(0.0) + 2.0 * away_r.fillna(0.0)

    team_cards = np.where(home_mask, home_cards, away_cards)
    opp_cards = np.where(home_mask, away_cards, home_cards)

    pk_for = pd.to_numeric(col_or_nan(out, "PK"), errors="coerce").fillna(0.0)
    pk_against = pd.to_numeric(col_or_nan(out, "PK_Allowed"), errors="coerce").fillna(
        0.0
    )

    raw_signal = (opp_cards - team_cards) + 2.0 * (pk_for - pk_against)
    raw_signal = pd.Series(raw_signal, index=out.index).fillna(0.0)

    std = raw_signal.std()
    out["_ref_signal_norm"] = (
        (raw_signal - raw_signal.mean()) / std if pd.notna(std) and std > 0 else 0.0
    )

    out["Referee_Bias_Score"] = (
        out.groupby(["_ref", "Team"])["_ref_signal_norm"]
        .transform(lambda s: s.shift(1).expanding().mean())
        .fillna(0.0)
    )

    relegation_mask = out["team_position"].fillna(10) >= 16
    title_mask = out["team_position"].fillna(10) <= 2
    situation_multiplier = np.where(relegation_mask | title_mask, 1.3, 1.0)
    out["Referee_Bias_Score"] = out["Referee_Bias_Score"] * situation_multiplier

    out.drop(columns=["_ref", "_ref_signal_norm"], inplace=True)
    log("Style-aware referee bias features attached")
    return out


# Build a motivation score from form, position, and fixture context.
def attach_dynamic_motivation(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(["Date", "Team"]).copy()
    motivation_scores = []

    def _position_component(team_pos: float) -> float:
        if pd.isna(team_pos):
            return 0.0
        if team_pos == 1:
            return 0.4
        if team_pos <= 5:
            return 0.25
        if team_pos <= 8:
            return 0.15
        if team_pos >= 16:
            return 0.5
        return 0.0

    for _, row in out.iterrows():
        current_date = row["Date"]
        history = out[out["Date"] < current_date]

        team = row["Team"]
        opp = row["Opponent"]
        team_pos = row.get("team_position", np.nan)
        opp_pos = row.get("opponent_position", np.nan)

        team_style = classify_team_style(history, team, window=10)
        opp_style = classify_team_style(history, opp, window=10)

        form = calculate_recent_form(history, team, window=5)
        form_score = 0.3 * form["points_per_game"]
        form_score += 0.05 * max(form["streak"], -3)

        position_score = _position_component(team_pos)
        opp_strength = calculate_opponent_strength(history, opp)
        opp_score = 0.15 * opp_strength
        fixture_score = 0.2 * calculate_fixture_importance(team_pos, opp_pos, row)
        style_interaction = opponent_style_interaction(team_style, opp_style)

        squad_depth = estimate_squad_depth(history, team, window=15)
        depth_factor = 0.9 + 0.2 * squad_depth
        boost = TEAM_BOOST_OVERRIDES.get(team.lower(), 0.0)

        total_motivation = (
            form_score
            + position_score
            + opp_score
            + fixture_score
            + style_interaction
            + boost
        ) * depth_factor
        motivation_scores.append(min(1.0, max(0.0, total_motivation)))

    out["Motivation_Score"] = motivation_scores
    log("Dynamic style-aware motivation scores calculated")
    return out

## Split Utility (Leakage-Safe Group Split)

In [74]:
# Make sure the validation and test splits are sensible.
def _validate_split_fractions(val_frac: float, test_frac: float) -> None:
    if val_frac <= 0 or test_frac <= 0:
        raise ValueError("val_frac and test_frac must both be > 0")
    if val_frac + test_frac >= 1.0:
        raise ValueError("val_frac + test_frac must be < 1.0")


# Build a stable group id for each fixture match-up.
def make_match_group_id(df: pd.DataFrame) -> pd.Series:
    t1 = df["Team"].str.lower().str.strip()
    t2 = df["Opponent"].str.lower().str.strip()
    lo = t1.where(t1 <= t2, t2)
    hi = t2.where(t1 <= t2, t1)
    date_str = (
        pd.to_datetime(df["Date"], errors="coerce").dt.strftime("%Y-%m-%d").fillna("NA")
    )
    return date_str + "|" + lo + "|" + hi


# Pick the dominant result type for a grouped match fixture.
def _group_strat_label(grp: pd.DataFrame) -> str:
    vc = grp["Result"].value_counts()
    if vc.empty:
        return "d"
    if "d" in vc.index and vc["d"] >= vc.max():
        return "d"
    return str(vc.idxmax())


# Split the data into train, validation, and test sets by match group.
def stratified_group_split(
    df: pd.DataFrame,
    val_frac: float = 0.20,
    test_frac: float = 0.10,
    random_state: int = 42,
):
    _validate_split_fractions(val_frac, test_frac)

    df = df.copy()
    df["_match_gid"] = make_match_group_id(df)

    grp_labels = (
        df.groupby("_match_gid")
        .apply(_group_strat_label)
        .reset_index()
        .rename(columns={0: "strat"})
    )

    g_train, g_temp = train_test_split(
        grp_labels,
        test_size=val_frac + test_frac,
        random_state=random_state,
        stratify=grp_labels["strat"],
    )

    relative_test = test_frac / (val_frac + test_frac)
    g_val, g_test = train_test_split(
        g_temp,
        test_size=relative_test,
        random_state=random_state,
        stratify=g_temp["strat"],
    )

    def _select(match_group_ids: pd.Series) -> pd.DataFrame:
        return (
            df[df["_match_gid"].isin(set(match_group_ids))]
            .drop(columns=["_match_gid"])
            .sample(frac=1, random_state=random_state)
            .reset_index(drop=True)
        )

    return (
        _select(g_train["_match_gid"]),
        _select(g_val["_match_gid"]),
        _select(g_test["_match_gid"]),
    )

## Training Workflow Functions

In [75]:
# Clean the raw league columns into a model-ready base table.
def _normalize_base_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out["Result"] = out["Result"].astype(str).str.lower().str.strip()
    out["Team"] = out["Team"].map(normalize_team)
    out["Opponent"] = out["Opponent"].map(normalize_team)
    out["xG"] = pd.to_numeric(out["xG"], errors="coerce")
    out["xGA"] = pd.to_numeric(out["xGA"], errors="coerce")
    out["home_advantage"] = (
        parse_home_away(out).replace({"home": "h", "away": "a"}).fillna("a")
    )
    return out


# Add helper columns used to build rolling features.
def _add_rolling_helper_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["_scored"] = pd.to_numeric(col_or_nan(out, "Scored"), errors="coerce")
    out["_conceded"] = pd.to_numeric(col_or_nan(out, "Conceded"), errors="coerce")
    out["_win_val"] = out["Result"].map({"w": 1.0, "d": 0.5, "l": 0.0})
    out["_ppda"] = pd.to_numeric(
        out.get("PPDA", pd.Series(dtype=float)), errors="coerce"
    )
    if "xpts" in out.columns:
        out["_xpts"] = pd.to_numeric(out["xpts"], errors="coerce")
    return out


# Keep only rows that can be used for training.
def _filter_trainable_rows(df: pd.DataFrame) -> pd.DataFrame:
    filtered = df[
        df["Date"].notna()
        & df["Result"].isin(["w", "d", "l"])
        & df["Team"].notna()
        & df["Opponent"].notna()
    ].copy()
    log(f"Rows after filtering: {len(filtered):,}")
    return filtered


# Load all competition fixture files and combine them.
def _load_competition_events(data_root: Path) -> pd.DataFrame:
    additional = data_root / "additional_data"
    return pd.concat(
        [
            load_competition_matches(additional / "champion_league.csv", "ucl"),
            load_competition_matches(additional / "europa_league.csv", "uel"),
            load_competition_matches(additional / "fa_cup.csv", "fa"),
            load_competition_matches(additional / "carabao.csv", "carabao"),
        ],
        ignore_index=True,
    )


# Remove temporary helper columns after rolling features are built.
def _drop_rolling_helper_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.drop(
        columns=["_scored", "_conceded", "_win_val", "_ppda", "_xpts"],
        errors="ignore",
        inplace=True,
    )
    return out


# Run the full feature engineering pipeline.
def load_engineered_dataset(data_root: Path) -> pd.DataFrame:
    additional = data_root / "additional_data"

    df = load_premier_league(data_root, additional)
    df = _normalize_base_columns(df)
    df = _add_rolling_helper_columns(df)
    df = _filter_trainable_rows(df)

    comp_df = _load_competition_events(data_root)
    df = attach_fatigue(df, comp_df)

    df = build_rolling_features(df)
    df = _drop_rolling_helper_columns(df)

    pos_map = load_position_map(data_root / "league_position_after20.csv")
    df = attach_position_features(df, pos_map)

    df = attach_h2h_features(df)
    df, _ = attach_context_features(df)
    df = attach_referee_features(df)
    df = attach_dynamic_motivation(df)
    df = attach_style_engineered_features(df)

    return df


# Decide which engineered columns are used as model features.
def select_feature_columns(df: pd.DataFrame):
    w = ROLLING_WINDOW
    base_numeric_cols = ["xG", "xGA"]
    rolling_keep_cols = [
        f"rolling_xpts_{w}",
        f"rolling_ppda_{w}",
        f"rolling_scored_{w}",
        f"rolling_conceded_{w}",
        "rolling_xG_style_adj",
        "rolling_xGA_style_adj",
    ]
    context_keep_cols = [
        "Possession_Style_Weighted",
        "SCA_Style_Weighted",
        "Dribbles_Style_Weighted",
        "Final_Third_Style_Weighted",
        "Possession_Style_Delta",
        "Final_Third_Entries_Allowed",
        "Aerial_Battles_Won_Pct",
        "Save_Pct",
        "PPDA",
        "Allowed_PPDA",
    ]
    psych_cols = ["Referee_Bias_Score", "Motivation_Score", "referee_style_impact"]
    interaction_cols = ["xG_diff", "form_vs_strength", "momentum"]

    numeric_cols = [
        c
        for c in (
            base_numeric_cols
            + ["h2h_win_rate"]
            + rolling_keep_cols
            + context_keep_cols
            + psych_cols
            + interaction_cols
        )
        if c in df.columns
    ]
    cat_cols = ["home_advantage"]
    all_feat_cols = numeric_cols + cat_cols

    log(
        f"Features: {len(all_feat_cols)} total (numeric={len(numeric_cols)}, categorical={len(cat_cols)})"
    )
    return numeric_cols, cat_cols, all_feat_cols


# Split the engineered dataframe into inputs and labels.
def split_features_and_target(
    df: pd.DataFrame,
    feature_cols: list[str],
    val_frac: float = 0.20,
    test_frac: float = 0.10,
):
    train_df, val_df, test_df = stratified_group_split(
        df, val_frac=val_frac, test_frac=test_frac
    )

    X_train, y_train = train_df[feature_cols], train_df["Result"]
    X_val, y_val = val_df[feature_cols], val_df["Result"]
    X_test, y_test = test_df[feature_cols], test_df["Result"]
    return X_train, y_train, X_val, y_val, X_test, y_test


# Print accuracy and classification metrics for a fitted model.
def print_model_report(
    model_name: str,
    model,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    include_feature_importance: bool = False,
):
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    sep = "=" * 62
    print(f"\n{sep}")
    print(f"  {model_name}")
    print(sep)
    print(f"  Validation Accuracy : {val_acc:.4f}")
    print(f"  Test Accuracy       : {test_acc:.4f}")
    print(f"\nValidation Report:\n{classification_report(y_val, y_val_pred, digits=4)}")
    print(f"Test Report:\n{classification_report(y_test, y_test_pred, digits=4)}")

    if include_feature_importance:
        clf = model.named_steps.get("clf")
        preprocess = model.named_steps.get("preprocess")
        if (
            clf is not None
            and preprocess is not None
            and hasattr(clf, "feature_importances_")
        ):
            feat_names = [
                name.split("__", 1)[-1] for name in preprocess.get_feature_names_out()
            ]
            importance_df = (
                pd.DataFrame(
                    {"feature": feat_names, "importance": clf.feature_importances_}
                )
                .sort_values("importance", ascending=False)
                .reset_index(drop=True)
            )
            print("Top 15 feature importances:")
            print(importance_df.head(15).to_string(index=False))

    return {
        "model": model_name,
        "validation_accuracy": val_acc,
        "test_accuracy": test_acc,
    }

## Model Builders and Training Run

In [76]:
# Build the preprocessing pipeline for numeric and categorical columns.
def build_preprocessor(
    numeric_features: list[str], categorical_features: list[str]
) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_features,
            ),
        ]
    )


# Build a random forest classification pipeline.
def build_random_forest_pipeline(
    numeric_features: list[str], categorical_features: list[str]
) -> Pipeline:
    preprocessor = build_preprocessor(numeric_features, categorical_features)
    clf = RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])


# Build a decision tree classification pipeline.
def build_decision_tree_pipeline(
    numeric_features: list[str], categorical_features: list[str]
) -> Pipeline:
    preprocessor = build_preprocessor(numeric_features, categorical_features)
    clf = DecisionTreeClassifier(
        max_depth=12, min_samples_leaf=5, class_weight="balanced", random_state=42
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])


# Build a logistic regression classification pipeline.
def build_logistic_regression_pipeline(
    numeric_features: list[str], categorical_features: list[str]
) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_features,
            ),
        ]
    )
    clf = LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=42, solver="lbfgs"
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])


# Build an XGBoost classification pipeline.
def build_xgboost_pipeline(
    numeric_features: list[str], categorical_features: list[str]
) -> Pipeline:
    preprocessor = build_preprocessor(numeric_features, categorical_features)
    clf = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])


df = load_engineered_dataset(DATA_ROOT)
numeric_cols, cat_cols, all_feat_cols = select_feature_columns(df)
X_train, y_train, X_val, y_val, X_test, y_test = split_features_and_target(
    df, all_feat_cols
)

rf_model = build_random_forest_pipeline(numeric_cols, cat_cols)
dt_model = build_decision_tree_pipeline(numeric_cols, cat_cols)
lr_model = build_logistic_regression_pipeline(numeric_cols, cat_cols)

rf_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)
lr_model.fit(X_train, y_train)

results = []
results.append(
    print_model_report(
        "RandomForestClassifier n_estimators=200",
        rf_model,
        X_val,
        y_val,
        X_test,
        y_test,
        include_feature_importance=True,
    )
)
results.append(
    print_model_report("DecisionTreeClassifier", dt_model, X_val, y_val, X_test, y_test)
)
results.append(
    print_model_report(
        "LogisticRegression (linear baseline)", lr_model, X_val, y_val, X_test, y_test
    )
)

# Train the XGBoost model and add its scores to the results table.
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

xgb_model = build_xgboost_pipeline(numeric_cols, cat_cols)
xgb_model.fit(X_train, y_train_enc)

y_val_pred_enc = xgb_model.predict(X_val)
y_test_pred_enc = xgb_model.predict(X_test)

val_acc = accuracy_score(y_val_enc, y_val_pred_enc)
test_acc = accuracy_score(y_test_enc, y_test_pred_enc)

print("\n" + "=" * 62)
print("  XGBoostClassifier")
print("=" * 62)
print(f"  Validation Accuracy : {val_acc:.4f}")
print(f"  Test Accuracy       : {test_acc:.4f}")
print(
    f"\nValidation Report:\n {classification_report(
        y_val_enc,
        y_val_pred_enc,
        target_names=label_encoder.classes_,
        digits=4)}"
)
print(
    f"Test Report:\n { classification_report(
        y_test_enc,
        y_test_pred_enc,
        target_names=label_encoder.classes_,
        digits=4
    )}"
)

results.append(
    {
        "model": "XGBoostClassifier",
        "validation_accuracy": val_acc,
        "test_accuracy": test_acc,
    }
)

[INFO] Loaded additional_data/premier_league.csv
[INFO] Raw rows after concat + dedup: 3,076
[INFO] Rows after filtering: 3,076
[INFO] Fatigue scores attached from 920 competition rows
[INFO] Rolling features built (window=5)
[INFO] H2H win rate attached
[INFO] Context features: ['Possession', 'Shot_Creating_Actions', 'Successful_Dribbles', 'Final_Third_Entries', 'Final_Third_Entries_Allowed', 'Aerial_Battles_Won_Pct', 'Save_Pct', 'PPDA', 'Allowed_PPDA']
[INFO] Style-aware referee bias features attached
[INFO] Dynamic style-aware motivation scores calculated
[INFO] Style engineered features attached
[INFO] Features: 26 total (numeric=25, categorical=1)

  RandomForestClassifier n_estimators=200
  Validation Accuracy : 0.5519
  Test Accuracy       : 0.5974

Validation Report:
              precision    recall  f1-score   support

           d     0.3049    0.1471    0.1984       170
           l     0.6151    0.7309    0.6680       223
           w     0.5651    0.6816    0.6179       2

## Results Summary

In [77]:
results_df = (
    pd.DataFrame(results)
    .sort_values("test_accuracy", ascending=False)
    .reset_index(drop=True)
)
results_df

,model,validation_accuracy,test_accuracy
0,RandomForestClassifier n_estimators=200,0.551948,0.597403
1,LogisticRegression (linear baseline),0.558442,0.571429
2,XGBoostClassifier,0.564935,0.564935
3,DecisionTreeClassifier,0.516234,0.542208
